# Spoken Language Processing - Instituto Superior Técnico
### Laboratory Assignment 2 - Automatic Age Estimation Challenge
<!--[image](imgs/lab2_slp_banner.png)-->
<img src="imgs/lab2_slp_banner.png" alt="drawing" width="400"/>


# WEEK 2 - Using pre-trained models


During this week, students will implement two modern systems for age regression based on:
- speaker representations (utterance-based) obtained with an x-vector model (`lab2_xvec.ipynb` notebook);
- speech representations (frame-based) obtained with a self-supervised learning (SSL) pre-trained model (this notebook).

In both cases, students are encouraged to explore different feature configurations and alternative downstream models.

## Before starting

Let's import some modules and make some definitions:

Like in the previous Notebooks, you need to upload pf_tools.py and requirements.txt if you are working on Google Colab. Otherwise, you should skip or delete the following code cell:

In [14]:
import os
import csv
import pickle
import numpy as np
import librosa
import torch
from torch import nn

from pf_tools import CheckThisCell, SLPdata

# We removed the SpeechBrain imports here because they are only needed 
# for the x-vector notebook (lab2_xvec.ipynb), not this SSL notebook!
from sklearn.svm import LinearSVC, SVR
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

GENDER_CLASSES = ('F',  'M')
GEN2ID = {'F':0, 'M':1}
ID2GEN = dict((GEN2ID[k],k)for k in GEN2ID)


Like in week1, the audio data is expected to be in a folder with the following format:

```
ets_data/
├── train/
│   └── audio/
│       └──wav files
│   └── info.csv
│
└── train100/
    └── audio/
        └──wav files
    └── info.csv
...
```

You must already have this from the previous week, so you can set-up your data directory:

In [15]:
import os

CWD = os.getcwd() 
DATADIR = os.path.abspath(os.path.join(CWD, '..', 'data'))

# Create the data directory if it doesn't exist
if not os.path.isdir(DATADIR):
    os.makedirs(DATADIR)
    print(f"Created directory: {DATADIR}")

print(f'Current working directory is set to: {CWD}')   
print(f'Your LAB2 data folder is: {DATADIR}')

Current working directory is set to: c:\Users\luis\OneDrive\IST\Mestrado\1ºAno\2semestre\4periodo\PF\slp_lab2\src
Your LAB2 data folder is: c:\Users\luis\OneDrive\IST\Mestrado\1ºAno\2semestre\4periodo\PF\slp_lab2\data


If you need to download again the data, you can run the following cell:

In [ ]:
raise CheckThisCell

os.chdir(DATADIR)

# download train
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/train.tgz
!tar -xzvf train.tgz

#download train100
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/train_small.tgz
!tar -xzvf train_small.tgz

#download dev
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/dev.tgz
!tar -xzvf dev.tgz

#download evl
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/evl.tgz
!tar -xzvf evl.tgz

os.chdir(CWD)

## Using self-supervised pre-trained models (SSL)

The goal of this part of the laboratory is to expose students to modern tools and methods for speech classification.
In particular, we will explore self-supervised learning (SSL) models (available at [HuggingFace](https://huggingface.co/)) to build a age regression system.

We will explore two approaches:

- A system that follows the same structure of the last x-vector  system: a pre-trained model (typically referred to as upstream model) to extract features followed by a decoupled downstream neural model. There will be no adaptation of the upstream model. This is sometimes refered in the literature as *linear probing*.

- **OPTIONAL** A system that  consists of a regression head on top of the  SSL model, but in this case, the entire network will be fine-tuned (this process can be significantly slower than any previous system we trained so far).

Notice that when comparing SSL features with x-vector features there are two fundamental differences that impact our regressor:
1. SSL models are pre-trained in a self-supervised way, thus, potentially with larger amounts of data;
2. SSL models produce features at the frame-level.

In this part of the lab, students are  expected to *play* with the different upstream models to build the best possible age estimation system.

In particular, students are encouraged to explore and discover which of the available SSL models can be a better candidate for their regression system. Potential candidates are [HuBERT](https://huggingface.co/docs/transformers/model_doc/hubert), [wav2vec2](https://huggingface.co/docs/transformers/model_doc/wav2vec2), and [WavLM](https://huggingface.co/docs/transformers/model_doc/wavlm) among others. Note that using a large SSL model will make the feature extraction process quite slow and the fine-tuning **VEEEERY SLOW**. Notice also that there may be different versions available of each model, trained with different amounts of data or with increased number of parameters.

## 1. SSL model as a (frozen) feature extractor

### 1.1 Extracting SSL features

The following code snipet shows how to load an SSL model from huggingface, load an audio file and preprocess it.

In [ ]:
import librosa
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model, logging as tf_logging

# Define model and data paths
model_name = "facebook/wav2vec2-base"
audiofile = f'{DATADIR}/train_small/wav/00834c0e904d40eda496e55010acebc5.wav'

# 1. Load the processor (handles audio normalization/feature extraction)
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

# 2. Silence the warning logs specifically during model loading
tf_logging.set_verbosity_error()
model = Wav2Vec2Model.from_pretrained(model_name)
tf_logging.set_verbosity_warning() # Restore standard logging for the rest of your script

# 3. Load and preprocess the audio file
audio, _ = librosa.load(audiofile, sr=16000, duration=10.0, mono=True)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)



Then, we will pass the processed input through the model. We will use as features the activations of the last hidden layer. Later, you can play with the specific layer used. Researchers have shown that some middle layers can contain more information for tasks similar to ours.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    last_hidden_states = outputs.hidden_states[-1]  # Get the last hidden states
    print(inputs['input_values'].shape)  # (batch_size, sequence_length)
    print(last_hidden_states.shape)  # (batch_size, sequence_length, hidden_size)



Notice the dimension of the input and the output. What is the relation between them? What is the frame rate of the SSL model?

Use this example to write a function that takes as arguments an audio filename, an SSL preprocessor and an SSL model and returns a numpy array of dimension (1xD) (you can compute the mean to reduce the time dimension).

In [ ]:
def extract_wav2vec(filename, processor, model, duration=10.0, num_layer=-1):
    """
    Extract self-supervised embedding from an audio file.
    Returns numpy array of shape (1, D).
    """

    signal, sr = librosa.load(filename, sr=16000, duration=duration)
    inputs = processor(signal, sampling_rate=16000, return_tensors="pt")
    
    # Move inputs to the same device as the model (e.g., GPU) to prevent crashes
    device = next(model.parameters()).device
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    # 3. Forward pass through the SSL model
    with torch.no_grad():
        # Magic parameter: output_hidden_states=True allows us to see inside the layers
        outputs = model(**inputs, output_hidden_states=True)
        
    hidden_states = outputs.hidden_states[num_layer] 
    # Shape of hidden_states: (batch_size, sequence_length, hidden_size)
    
    embedding = hidden_states.mean(dim=1) 
    # Shape is now: (batch_size, hidden_size) --> e.g., (1, 768)
    
    embedding = embedding.cpu().numpy()
    
    return embedding

# This must return a numpy array of shape (1, 768)
# Using an audio file from 'train_small' for consistency with previous cells
feat = extract_wav2vec(f'{DATADIR}/train_small/wav/00834c0e904d40eda496e55010acebc5.wav', processor, model)
print(feat.shape, type(feat))


The same way as we did with all the previous systems, we will define the feature extraction configuration using a dictionary:

In [ ]:
#model_name = "facebook/wav2vec2-base"
#processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
#model = Wav2Vec2Model.from_pretrained(model_name)

transform = {}
transform['wav2vec2'] = {
                    'audio_transform':
                        lambda x :
                            extract_wav2vec(x,
                            processor = processor,
                            model = model
                            ),
                    'chunk_transform': None,
                    'chunk_size': 0,
                    'chunk_hop':0
                }

transform['wav2vec2_7thlayer'] = {
                    'audio_transform':
                        lambda x :
                            extract_wav2vec(x,
                            processor = processor,
                            model = model,
                            num_layer=7
                            ),
                    'chunk_transform': None,
                    'chunk_size': 0,
                    'chunk_hop':0
                }

model_name = "audeering/wav2vec2-large-robust-24-ft-age-gender"
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name)

transform['audeering_w2v2_layer14'] = {
                    'audio_transform': 
                        lambda x: 
                            extract_wav2vec(x,
                            processor=processor,
                            model=model,
                            num_layer=14  
                            ),
                    'chunk_transform': None,
                    'chunk_size': 0,
                    'chunk_hop': 0
                }

In [ ]:

# Download and feature extract
trainset = 'train_small'
transform_id = 'audeering_w2v2_layer14'

slp_partitions = {}
# for partition in ('train', 'train100', 'dev', 'evl'):
for partition in ('train_small', 'dev', 'evl'):
    slp_partitions[partition] = SLPdata(DATADIR, partition,
                    transform_id=transform_id,
                    audio_transform=transform[transform_id]['audio_transform'],
                    chunk_transform=transform[transform_id]['chunk_transform'],
                    chunk_size=transform[transform_id]['chunk_size'],
                    chunk_hop=transform[transform_id]['chunk_hop']
                    )


And instantiate the SLP class for all the data partitions to apply the feature extraction. **WARNING** This can be very slow depending on the model and the available computational resources.

# FUSED SSL

In [16]:
# ================================================================
# CELL 1: LOAD ALL MODELS
# ================================================================
import os
import numpy as np
import torch
import librosa
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model, WavLMModel

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading Audeering...")
processor_audeering = Wav2Vec2FeatureExtractor.from_pretrained("audeering/wav2vec2-large-robust-24-ft-age-gender")
model_audeering = Wav2Vec2Model.from_pretrained("audeering/wav2vec2-large-robust-24-ft-age-gender", use_safetensors=True).to(device)
model_audeering.eval()

print("Loading WavLM Base+...")
processor_wavlm = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
model_wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus", use_safetensors=True).to(device)
model_wavlm.eval()

print("All models loaded.")

Using device: cuda:0
Loading Audeering...


Loading weights: 100%|██████████| 422/422 [00:00<00:00, 9413.82it/s]


Loading WavLM Base+...


Loading weights: 100%|██████████| 248/248 [00:00<00:00, 9721.92it/s]


All models loaded.


In [17]:
# ================================================================
# CELL 2: DEFINE ALL EXTRACTION FUNCTIONS
# ================================================================

def extract_single_layer(filename, processor, model, layer):
    """One layer from any SSL model → (1, hidden_size)"""
    speech, _ = librosa.load(filename, sr=16000)
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = model(inputs.input_values.to(device), output_hidden_states=True)
    pooled = outputs.hidden_states[layer].mean(dim=1).squeeze(0)
    return pooled.cpu().numpy().reshape(1, -1)


def extract_all_layers(filename, processor, model):
    """All layers from any SSL model → (1, num_layers * hidden_size)"""
    speech, _ = librosa.load(filename, sr=16000)
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = model(inputs.input_values.to(device), output_hidden_states=True)
    pooled_layers = [hs.mean(dim=1).squeeze(0).cpu().numpy() for hs in outputs.hidden_states]
    return np.concatenate(pooled_layers).reshape(1, -1)


print("Extraction functions defined.")

Extraction functions defined.


In [20]:
# ================================================================
# CELL 3: REGISTER ALL TRANSFORMS
# ================================================================
transform = {}

# ── Single layer ─────────────────────────────────────────────────
transform['audeering_layer18'] = {
    'audio_transform': lambda f: extract_single_layer(f, processor_audeering, model_audeering, layer=18),
    'chunk_transform': None, 'chunk_size': 0, 'chunk_hop': 0
}
transform['wavlm_layer8'] = {
    'audio_transform': lambda f: extract_single_layer(f, processor_wavlm, model_wavlm, layer=8),
    'chunk_transform': None, 'chunk_size': 0, 'chunk_hop': 0
}

# ── All layers (for WeightedLayerPoolingMLP) ─────────────────────
# Audeering: 25 layers × 1024 = 25600 dims
# WavLM:     13 layers × 768  = 9984  dims
transform['audeering_all_layers'] = {
    'audio_transform': lambda f: extract_all_layers(f, processor_audeering, model_audeering),
    'chunk_transform': None, 'chunk_size': 0, 'chunk_hop': 0
}
transform['wavlm_all_layers'] = {
    'audio_transform': lambda f: extract_all_layers(f, processor_wavlm, model_wavlm),
    'chunk_transform': None, 'chunk_size': 0, 'chunk_hop': 0
}

# ── Fusion: Audeering layer 18 + WavLM layer 8 → 1792 dims ──────
transform['fused_aud_wavlm'] = {
    'audio_transform': lambda f: np.concatenate([
        extract_single_layer(f, processor_audeering, model_audeering, layer=18).flatten(),
        extract_single_layer(f, processor_wavlm,     model_wavlm,     layer=8 ).flatten()
    ]).reshape(1, -1),
    'chunk_transform': None, 'chunk_size': 0, 'chunk_hop': 0
}

# ── Fusion: Audeering all layers + WavLM all layers → 35584 dims ─
transform['fused_all_layers'] = {
    'audio_transform': lambda f: np.concatenate([
        extract_all_layers(f, processor_audeering, model_audeering).flatten(),
        extract_all_layers(f, processor_wavlm,     model_wavlm    ).flatten()
    ]).reshape(1, -1),
    'chunk_transform': None, 'chunk_size': 0, 'chunk_hop': 0
}

print("Registered transforms:")
for k in transform:
    print(f"  {k}")

Registered transforms:
  audeering_layer18
  wavlm_layer8
  audeering_all_layers
  wavlm_all_layers
  fused_aud_wavlm
  fused_all_layers


In [22]:
# ================================================================
# CELL 4: RUN EXTRACTION
# ================================================================

transform_id = 'audeering_all_layers'  # ← change this

slp_partitions = {}
print(f"\nExtracting: '{transform_id}'")

for partition in ('train', 'dev', 'evl'):
    print(f"  -> {partition}...")
    slp_partitions[partition] = SLPdata(
        DATADIR, partition,
        transform_id=transform_id,
        audio_transform=transform[transform_id]['audio_transform'],
        chunk_transform=transform[transform_id]['chunk_transform'],
        chunk_size=transform[transform_id]['chunk_size'],
        chunk_hop=transform[transform_id]['chunk_hop']
    )

print("Done.")


Extracting: 'audeering_all_layers'
  -> train...


c:\Users\luis\OneDrive\IST\Mestrado\1ºAno\2semestre\4periodo\PF\slp_lab2\src\pf_tools.py:108: UserWarning: The feature directory already exists, and no new feature extraction will be performed.
  warnings.warn("The feature directory already exists, and no new feature extraction will be performed.")


  -> dev...
  -> evl...
Done.


c:\Users\luis\OneDrive\IST\Mestrado\1ºAno\2semestre\4periodo\PF\slp_lab2\src\pf_tools.py:108: UserWarning: The feature directory already exists, and no new feature extraction will be performed.
  warnings.warn("The feature directory already exists, and no new feature extraction will be performed.")
c:\Users\luis\OneDrive\IST\Mestrado\1ºAno\2semestre\4periodo\PF\slp_lab2\src\pf_tools.py:108: UserWarning: The feature directory already exists, and no new feature extraction will be performed.
  warnings.warn("The feature directory already exists, and no new feature extraction will be performed.")


### 1.2.1 Training the SVR

In a similar way as our `x-vector` work, we will try to use the SVR with SSL features.


In [23]:
from pf_tools import prepare_slp_data

#   Concatenate all data and labels
#   Each row corresponds to a file
#   We store the data, labels and file identifiers of each partition in dictionaries
#     with the partition name as key


gender_label_pos = 0
age_label_pos = 1

data, labels_gender, labels_age, fileids = {}, {}, {}, {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train', 'dev', 'evl'):
    data_and_labels = prepare_slp_data(slp_partitions[partition])
    print(f'Partition: {partition}')
    print(f'Number of samples: {data_and_labels["data"].shape[0]}')
    print(f'Number of features: {data_and_labels["data"].shape[1]}')
    print(f'Number of labels: {len(np.unique(data_and_labels["label"][:,gender_label_pos]))}')
    print(f'Number of identifiers (samples): {len(np.unique(data_and_labels["identifiers"]))}')
    print('---')
    data[partition] = data_and_labels['data']
    labels_gender[partition] = data_and_labels['label'][:,gender_label_pos]
    labels_age[partition] = data_and_labels['label'][:,age_label_pos]
    fileids[partition] = data_and_labels['identifiers']



Partition: train
Number of samples: 3237
Number of features: 25600
Number of labels: 2
Number of identifiers (samples): 3237
---
Partition: dev
Number of samples: 117
Number of features: 25600
Number of labels: 2
Number of identifiers (samples): 117
---
Partition: evl
Number of samples: 145
Number of features: 25600
Number of labels: 1
Number of identifiers (samples): 145
---


In [ ]:
from sklearn.svm import SVR
from pf_tools import save_model

trainset = 'train_small'

# Train a SVR
model = SVR(kernel='linear') ### <---- a linear SVR
model.fit(data[trainset], labels_age[trainset])  ## <---- train MODEL

model_id = save_model(model, f'svr_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/')

# Predict the dev and evl sets
dev_results = model.predict(data['dev']) #  Predict dev
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp':dev_results, 'fileids':fileids['dev']}, open(filename, 'wb'))

evl_results = model.predict(data['evl']) #  Predict evl
filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp':evl_results, 'fileids':fileids['evl']}, open(filename, 'wb'))


In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error

ref, hyp = labels_age['dev'], dev_results

print(f'Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}')
print(f'Mean Squared Error: {mean_squared_error(ref, hyp):.2f}')

### 1.2.2 Training the downstream model

Now let's try to train our regression task with SSL features with a simple neural model like in the last `x-vector` model.

In [ ]:
feat_dim = feat.shape[1] ### <--- Updated here with actual feature dimension

# Define a simple linear regression model
class LinearRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(LinearRegressor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1) # Output a single value for regression
        )

    def forward(self, x):
        return self.model(x)

model = LinearRegressor(input_dim=feat_dim, hidden_dim=100)

In [ ]:
from pf_tools import train_nn, save_model

train_nn(model, slp_partitions[trainset], slp_partitions['dev'], batch_size=16, epochs=200, lr=0.0005)

model_id = save_model(model, f'nnet_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/')

### 1.3 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pf_tools import predict_nn

# Predict the dev set
hyp, ref, files = predict_nn(model, slp_partitions['dev'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp':hyp, 'fileids':files}, open(filename, 'wb'))

# Report the results
print(f'Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}')
print(f'Mean Squared Error: {mean_squared_error(ref, hyp):.2f}')

# Predict the evl set
hyp, ref, files = predict_nn(model, slp_partitions['evl'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp':hyp, 'fileids':files}, open(filename, 'wb'))


At this point, you can explore different SSL model versions and layer configurations for feature extraction and alternative neural model architectures. You can also generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/8d80747e0c474688a83024aabdfe1ab0):

In [ ]:
from pf_tools import create_submission_file

students_group = '00' # <--- CHANGE THIS ACCORDINGLY

# model_id = 'svm_spkrec-ecapa-voxceleb_2025-05-04_19:10:21'
model_id_short = 'nnet_wv2v2'

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)

# ULTIMATE TEST SECTION

In [24]:
# ================================================================
# CELL 2: TUNED RIDGE REGRESSOR PIPELINE
# ================================================================
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedKFold

# Configuration Setup
TRAIN_PARTITION = 'train' # Change to 'train' when ready!
X_full = data[TRAIN_PARTITION]
y_full = labels_age[TRAIN_PARTITION].astype(float)
X_dev = data['dev']
y_dev_np = labels_age['dev'].astype(float)

# Stratification Layout
bins = [20, 30, 40, 50, 60, 70]
y_binned = np.digitize(y_full, bins)
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

print(f"\n{'='*75}\n RUNNING TUNED RIDGE REGRESSOR\n{'='*75}")
ridge_cv_maes = []
ridge_oof_preds = np.zeros(len(y_full))
ridge_dev_preds = np.zeros((N_SPLITS, len(y_dev_np)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_binned)):
    X_tr_f, y_tr_f = X_full[train_idx], y_full[train_idx]
    X_va_f, y_va_f = X_full[val_idx], y_full[val_idx]
    
    # Grid testing a wide scope of alpha parameters
    model = make_pipeline(
        RobustScaler(),
        RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 5000.0, 10000.0], scoring='neg_mean_absolute_error')
    )
    model.fit(X_tr_f, y_tr_f)
    
    val_hyp = model.predict(X_va_f)
    ridge_oof_preds[val_idx] = val_hyp
    ridge_cv_maes.append(mean_absolute_error(y_va_f, val_hyp))
    
    ridge_dev_preds[fold, :] = np.clip(model.predict(X_dev), 18, 80)
    best_alpha = model.named_steps['ridgecv'].alpha_
    print(f"Fold {fold + 1} | Val MAE: {ridge_cv_maes[-1]:.3f} | Selected Alpha: {best_alpha}")

ridge_mean_cv = np.mean(ridge_cv_maes)
ridge_final_dev = np.mean(ridge_dev_preds, axis=0)
ridge_blind_mae = mean_absolute_error(y_dev_np, ridge_final_dev)

print(f"\n-> Ridge Overall CV MAE: {ridge_mean_cv:.3f} | Blind Dev MAE: {ridge_blind_mae:.3f}\n")


 RUNNING TUNED RIDGE REGRESSOR
Fold 1 | Val MAE: 5.436 | Selected Alpha: 5000.0
Fold 2 | Val MAE: 5.137 | Selected Alpha: 1000.0
Fold 3 | Val MAE: 5.529 | Selected Alpha: 5000.0
Fold 4 | Val MAE: 5.520 | Selected Alpha: 1000.0
Fold 5 | Val MAE: 5.456 | Selected Alpha: 1000.0

-> Ridge Overall CV MAE: 5.416 | Blind Dev MAE: 5.367



In [ ]:
# ================================================================
# MASTER GRID SWEEP — GPU, Scaling, Gap Tracking & CSV Export
# ================================================================
import itertools
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd # <--- Added for CSV export

# Ensure device is defined natively
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# THE FULL GRID DICTIONARY (144 Combinations)
configs = {
    'loss':         ['l1', 'mse'],                    
    'lr':           [1e-3, 1e-4],               
    'dropout':      [(0.4, 0.3, 0.2), (0.3, 0.2, 0.1)],  
    'hidden':       [(1024, 512, 256), (512, 256, 128), (256, 128, 64), (128, 128, 64)], 
    'weight_decay': [1e-3, 1e-4],                    
    'batch_size':   [32, 64]                          
}


class WeightedLayerPoolingMLP(nn.Module):
    def __init__(self, num_layers, hidden_size, hidden_dims=(512,256,128), dropouts=(0.3,0.2,0.1)):
        super().__init__()
        self.num_layers  = num_layers
        self.hidden_size = hidden_size
        self.layer_weights = nn.Parameter(torch.ones(num_layers))

        layers = []
        in_dim = hidden_size
        for out_dim, drop in zip(hidden_dims, dropouts):
            layers += [
                nn.Linear(in_dim, out_dim),
                nn.BatchNorm1d(out_dim),
                nn.ReLU(),
                nn.Dropout(drop),
            ]
            in_dim = out_dim
        layers.append(nn.Linear(in_dim, 1))
        self.regressor = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, self.num_layers, self.hidden_size)
        w = torch.softmax(self.layer_weights, dim=0)
        x = (x * w.unsqueeze(0).unsqueeze(-1)).sum(dim=1)
        return self.regressor(x).squeeze(-1)


def run_config(loss_name, lr, dropouts, hidden_dims, wd, batch_size,
               X_full, y_full, X_dev, y_dev_np,
               num_layers=25, hidden_size=1024,
               n_splits=5, epochs=80):

    criterion = nn.L1Loss() if loss_name == 'l1' else nn.MSELoss()
    bins     = [20, 30, 40, 50, 60, 70]
    y_binned = np.digitize(y_full, bins)
    skf      = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    cv_maes   = []
    dev_preds = np.zeros((n_splits, len(y_dev_np)))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_binned)):
        X_tr, y_tr = X_full[tr_idx], y_full[tr_idx]
        X_va, y_va = X_full[va_idx], y_full[va_idx]

        # 🚨 FEATURE NORMALIZATION 🚨
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)
        X_dev_scaled = scaler.transform(X_dev) 

        # Send Model to GPU
        model = WeightedLayerPoolingMLP(num_layers, hidden_size, hidden_dims, dropouts).to(device)

        opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)

        loader = DataLoader(
            TensorDataset(
                torch.tensor(X_tr_scaled, dtype=torch.float32),
                torch.tensor(y_tr, dtype=torch.float32)
            ), batch_size=batch_size, shuffle=True
        )

        model.train()
        
        # --- EARLY STOPPING TRACKERS ---
        best_loss = float('inf')
        early_stop_patience = 15
        epochs_without_improvement = 0
        
        for epoch in range(epochs):
            epoch_loss = 0
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                
                opt.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                opt.step()
                epoch_loss += loss.item()
                
            sched.step(epoch_loss)
            
            if epoch_loss < best_loss:
                best_loss = epoch_loss
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                
            if epochs_without_improvement >= early_stop_patience:
                break # Silently break to keep terminal clean

        model.eval()
        with torch.no_grad():
            va_tensor = torch.tensor(X_va_scaled, dtype=torch.float32).to(device)
            dev_tensor = torch.tensor(X_dev_scaled, dtype=torch.float32).to(device)
            
            va_p  = model(va_tensor).cpu().numpy()
            dev_p = model(dev_tensor).cpu().numpy()

        cv_maes.append(mean_absolute_error(y_va, va_p))
        dev_preds[fold] = np.clip(dev_p, 18, 80)

    # Calculate final scores for the return block
    final_dev = np.mean(dev_preds, axis=0)
    dev_mae = mean_absolute_error(y_dev_np, final_dev)
    cv_mae = np.mean(cv_maes)

    return {
        'cv_mae':  cv_mae,
        'dev_mae': dev_mae,
        'gap':     abs(dev_mae - cv_mae),
        'raw_dev_preds': dev_preds
    }


# ── Dynamically generate ALL combinations ──────────────────────
keys, values = zip(*configs.items())
all_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
results_list = []

print(f"Starting FULL SWEEP ({len(all_combinations)} configurations) using device: {device}\n")

for i, kwargs in enumerate(all_combinations):
    label = f"loss={kwargs['loss']} lr={kwargs['lr']} drop={kwargs['dropout'][0]} hid={kwargs['hidden'][0]} wd={kwargs['weight_decay']} bs={kwargs['batch_size']}"
    
    r = run_config(
        loss_name=kwargs['loss'], 
        lr=kwargs['lr'], 
        dropouts=kwargs['dropout'], 
        hidden_dims=kwargs['hidden'], 
        wd=kwargs['weight_decay'], 
        batch_size=kwargs['batch_size'],
        X_full=X_full, y_full=y_full, X_dev=X_dev, y_dev_np=y_dev_np
    )

    # Append highly structured data for the CSV
    results_list.append({
        'Loss': kwargs['loss'],
        'Learning_Rate': kwargs['lr'],
        'Dropout_Base': kwargs['dropout'][0],
        'Hidden_Base': kwargs['hidden'][0],
        'Weight_Decay': kwargs['weight_decay'],
        'Batch_Size': kwargs['batch_size'],
        'CV_MAE': r['cv_mae'],
        'Dev_MAE': r['dev_mae'],
        'Gap': r['gap'],
        'Full_Config': label
    })
    
    # Print live progress
    print(f"[{i+1:>3}/{len(all_combinations)}] CV: {r['cv_mae']:.3f} | Dev: {r['dev_mae']:.3f} | Gap: {r['gap']:.3f}  -> {label}")


# ================================================================
# CSV EXPORT AND FINAL SUMMARY
# ================================================================
# Convert to DataFrame
df_results = pd.DataFrame(results_list)

# Sort the DataFrame by Gap (or Dev MAE, depending on preference)
df_results_sorted = df_results.sort_values(by=['Gap', 'Dev_MAE'], ascending=[True, True])

# Export to CSV
csv_filename = "hyperparameter_sweep_results.csv"
df_results_sorted.to_csv(csv_filename, index=False)

print(f"\n{'='*100}")
print(f"✅ SWEEP COMPLETE! All results perfectly formatted and saved to '{csv_filename}'")
print(f"{'='*100}\n")

# Print the top 5 most stable models to the terminal
print("🏆 TOP 5 MOST STABLE MODELS (Smallest Gap + Lowest Dev MAE):")
print(df_results_sorted[['Loss', 'Learning_Rate', 'Dropout_Base', 'Hidden_Base', 'Batch_Size', 'CV_MAE', 'Dev_MAE', 'Gap']].head(5).to_string(index=False))

In [ ]:
# ================================================================
# FINAL SUBMISSION: GOLDEN CONFIGURATION ENSEMBLE
# ================================================================
import os
import pickle
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Import submission tool based on your snippet
from pf_tools import create_submission_file 

# -------------------------------------------------------------------
# 1. THE GOLDEN CONFIGURATION
# -------------------------------------------------------------------
cfg = {
    'loss': 'mse', 
    'lr': 0.001, 
    'drop': (0.3, 0.2, 0.1), 
    'hid': (128, 128, 64),  
    'wd': 0.001,  
    'bs': 32
}

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_id = "ssl_mlp_golden_ensemble" 
results_output_dir = f'{DATADIR}/{trainset}/models/{model_id}/'
os.makedirs(results_output_dir, exist_ok=True)

# -------------------------------------------------------------------
# 2. EXTRACT DATA
# -------------------------------------------------------------------
print("Extracting features using prepare_slp_data...")

train_prep = prepare_slp_data(slp_partitions['train'], collapse_samples=True)
X_train = train_prep['data']
y_train = np.array(train_prep['label'][:, 1], dtype=float) # Column 1 is Age

dev_prep = prepare_slp_data(slp_partitions['dev'], collapse_samples=True)
X_dev = dev_prep['data']
y_dev = np.array(dev_prep['label'][:, 1], dtype=float)
fileids_dev = dev_prep['identifiers']

evl_prep = prepare_slp_data(slp_partitions['evl'], collapse_samples=True)
X_evl = evl_prep['data']
fileids_evl = evl_prep['identifiers']

# -------------------------------------------------------------------
# 3. TRAIN THE 5-FOLD ENSEMBLE
# -------------------------------------------------------------------
print(f"Starting Golden Ensemble Training on {device}...")

criterion = nn.L1Loss() if cfg['loss'] == 'l1' else nn.MSELoss()
bins = [20, 30, 40, 50, 60, 70]
y_binned = np.digitize(y_train, bins)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dev_fold_preds = []
evl_fold_preds = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_binned)):
    print(f" -> Training Fold {fold+1}/5...")
    
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_va, y_va = X_train[va_idx], y_train[va_idx]
    
    # Scale features
    scaler = RobustScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_dev_scaled = scaler.transform(X_dev)
    X_evl_scaled = scaler.transform(X_evl)
    
    # Build Model with the exact winning architecture
    model = WeightedLayerPoolingMLP(
        num_layers=25, 
        hidden_size=1024, 
        hidden_dims=cfg['hid'], 
        dropouts=cfg['drop']
    ).to(device)
    
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
    
    loader = DataLoader(
        TensorDataset(torch.tensor(X_tr_scaled, dtype=torch.float32), 
                      torch.tensor(y_tr, dtype=torch.float32)), 
        batch_size=cfg['bs'], shuffle=True
    )
    
    model.train()
    best_loss, epochs_without_improvement = float('inf'), 0
    
    for epoch in range(80):
        epoch_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        sched.step(epoch_loss)
        
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= 15:
            break
            
    # Inference for this fold
    model.eval()
    with torch.no_grad():
        dev_t = torch.tensor(X_dev_scaled, dtype=torch.float32).to(device)
        evl_t = torch.tensor(X_evl_scaled, dtype=torch.float32).to(device)
        
        dev_p = model(dev_t).cpu().numpy()
        evl_p = model(evl_t).cpu().numpy()
        
        dev_fold_preds.append(np.clip(dev_p, 18, 80))
        evl_fold_preds.append(np.clip(evl_p, 18, 80))

# -------------------------------------------------------------------
# 4. ENSEMBLE AVERAGING & SAVING
# -------------------------------------------------------------------
# Average the 5 folds to create the ultimate prediction
final_hyp_dev = np.mean(dev_fold_preds, axis=0)
final_hyp_evl = np.mean(evl_fold_preds, axis=0)

# Save the results as Pickle exactly as the lab formatting expects
filename_dev = f'{results_output_dir}/dev.pkl'
pickle.dump({
    'hyp': final_hyp_dev.tolist(), 
    'fileids': fileids_dev, 
    'ref': y_dev.tolist()
}, open(filename_dev, 'wb'))

filename_evl = f'{results_output_dir}/evl.pkl'
pickle.dump({
    'hyp': final_hyp_evl.tolist(), 
    'fileids': fileids_evl, 
    'ref': [0] * len(fileids_evl) # Dummy list for evl
}, open(filename_evl, 'wb'))

print("\n=================================================")
print("INFERENCE COMPLETE AND RESULTS SAVED.")

# Report the Final Dev Results
dev_res = pickle.load(open(filename_dev, 'rb'))
hyp = dev_res['hyp']
ref = dev_res['ref']

print(f"GOLDEN ENSEMBLE Dev MAE: {mean_absolute_error(ref, hyp):.3f}")
print("=================================================")


In [ ]:
students_group = '19' # Changed to match your group number

model_id_short = 'ssl_mlp_golden_ensemble'
results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' 

create_submission_file(results_path, filename)
print(f"Final Submission CSV generated at: {filename}")

## 2. Fine-tuning the SSL model (optional)

The full potential of SSL models is attained when we finetune the complete model for the target task.

This can be a computationally demanding task and it is commonly necessary to use GPU training. For this reason, this last part of the lab2 assignment is completely optional.

Interested students are recommeded to either use Google Colab with GPU support or run this code in a machine with GPU. Notice also that it may be worth copying the code of the following cells to a Python script and run the program directly invocking the Python interpreter in a terminal.

In [ ]:
from pf_tools import extract_and_verify_data_for_ssl

from datasets import Dataset, DatasetDict, Audio
from transformers import Wav2Vec2FeatureExtractor
import pandas as pd

# --- Step 1: Load raw audio data into datasets.DatasetDict for fine-tuning ---
partitions_to_process = ('train_small', 'dev', 'evl')
raw_audio_datasets = DatasetDict()

for partition in partitions_to_process:
    audio_base_path = extract_and_verify_data_for_ssl(DATADIR, partition) # Get the actual audio base path

    partition_data_dir = os.path.join(DATADIR, partition)
    info_csv_path = os.path.join(partition_data_dir, 'info.csv')
    df_info = pd.read_csv(info_csv_path)

    audio_samples = []
    for index, row in df_info.iterrows():
        wav_filename = row['wav']
        full_audio_path = os.path.join(audio_base_path, wav_filename) # Use the discovered audio_base_path
        if os.path.exists(full_audio_path):
            audio_samples.append({
                "audio": {"path": full_audio_path},
                "label": float(row['age']), # Ensure label is float for regression
                "fileid": os.path.basename(wav_filename).split('.')[0] # Add fileid explicitly
            })
        else:
            print(f"Skipping missing audio file (expected at {full_audio_path}): {wav_filename}")

    current_dataset = Dataset.from_list(audio_samples)
    current_dataset = current_dataset.cast_column("audio", Audio(sampling_rate=16000))
    raw_audio_datasets[partition] = current_dataset
    print(f"Loaded {len(current_dataset)} raw audio samples for '{partition}' (fine-tuning).")



After loading the data partitions, we will apply the preprocessor of the model to each data sample and remove some unnecessary fields from the dataset instances:

In [ ]:

# --- Step 2: User's provided preprocess_function and its application ---

def preprocess_function(examples, processor, duration=10):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = processor(
        audio_arrays, sampling_rate=processor.sampling_rate, max_length=16000*duration, truncation=True
    )
    return inputs

model_name = "facebook/wav2vec2-base"
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

enc_data = {}
max_duration = 10 # <--- seconds
for partition in partitions_to_process:
    enc_data[partition] = raw_audio_datasets[partition].map(
        lambda x: preprocess_function(x, processor, max_duration),
        remove_columns=["audio", "fileid"], # Remove original audio and fileid columns as features are now processed and fileid is not needed for model input
        batched=True,
        batch_size=100 # Process in batches to save memory
    )
    # Rename 'label' to 'labels' as expected by HuggingFace Trainer for regression tasks
    enc_data[partition] = enc_data[partition].rename_column("label", "labels")
    # Set format for PyTorch training
    enc_data[partition].set_format("torch")

print("Data preparation for fine-tuning complete. Encoded data:")
for partition, ds in enc_data.items():
    print(f"  {partition}: {len(ds)} samples, features: {list(ds.features.keys())}")

We do some checkings on the labels, just to be sure that everyhing is OK:

In [ ]:
# For regression, we expect the 'labels' column in enc_data to contain numerical age values.
# We will check the type of the 'labels' column in the encoded dataset.

# Check if enc_data is populated and contains the 'train_small' partition
if 'train_small' in enc_data and enc_data['train_small'] is not None:
    if 'labels' in enc_data['train_small'].features:
        print(f"Label column type for enc_data['train_small']: {enc_data['train_small'].features['labels'].dtype}")
    else:
        print("Error: 'labels' column not found in enc_data['train_small'].")

    # Also print a sample of the labels to ensure they are numeric
    print(f"Sample labels from enc_data['train_small']: {enc_data['train_small']['labels'][:5]}")
else:
    print("Error: 'enc_data' is not correctly populated or 'train_small' partition is missing. Please ensure previous data preparation cells were run successfully.")

In [ ]:
from transformers import Wav2Vec2ForSequenceClassification


# This is a default pre-defined model that consists of a projection transformation,
# an average pooling and final regression layer.
# For regression, we typically use Wav2Vec2ForSequenceClassification with num_labels=1.
# Ensure you also set `problem_type="regression"` in TrainingArguments.
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1, # For regression, output is a single value
    # problem_type="regression" # This should be set in TrainingArguments for clarity
)

# model.freeze_base_model() ## <---- IF YOU FREEZE THE BASE MODEL, IT MEANS FINETUNING ONLY THE DOWNSTREAM REGRESSOR
model.freeze_feature_encoder() ## <--- IF YOU FREEZE THE FEATURE ENCODER, YOU ARE FREEZING THE CONVOLUTIONAL INITIAL LAYERS OF THE UPSTREAM MODEL

Finally, we set-up the training parameters and run the training process:

In [ ]:
from transformers import TrainingArguments, Trainer, Wav2Vec2ForSequenceClassification

import numpy as np
import evaluate
from sklearn.metrics import mean_squared_error
import torch.nn as nn # Import nn for MSELoss

model_id = 'w2v2base_age_regression_fe_freeze'  # <--- Change this to your model id/ output folder name

training_args = TrainingArguments(
    output_dir=model_id,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    num_train_epochs=100,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_mse", # Use Mean Squared Error for regression
    push_to_hub=False
    # problem_type="regression" # Explicitly set problem_type for regression - Removed, as it's not a TrainingArguments parameter
)

# Re-initialize the model here to ensure correct configuration for regression
# This ensures the model object used by the Trainer has problem_type="regression" set.
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1, # For regression, output is a single value
    problem_type="regression" # Crucial for the model to use regression loss (e.g., MSELoss)
)
model.freeze_feature_encoder() # Re-apply freezing if desired


# Custom metric computation for regression
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions.squeeze() # Remove single-dimensional entries
    references = eval_pred.label_ids
    return {"mse": mean_squared_error(references, predictions)}

# Define a custom Trainer class to override the compute_loss method
class CustomRegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None): # Added num_items_in_batch
        labels = inputs.pop("labels") # Extract labels before passing to model
        outputs = model(**inputs)    # Model will return logits as labels are not passed
        logits = outputs.logits.squeeze(-1) # Ensure logits match label shape for MSELoss
        loss_fct = nn.MSELoss()     # Use Mean Squared Error Loss
        loss = loss_fct(logits, labels.float()) # Ensure labels are float for MSELoss
        return (loss, outputs) if return_outputs else loss


trainer = CustomRegressionTrainer(
    model=model,
    args=training_args,
    train_dataset=enc_data['train_small'],
    eval_dataset=enc_data['dev'],
    # feature_extractor=processor, # Removed, as the dataset is already preprocessed
    compute_metrics=compute_metrics
    # compute_loss is now handled by the custom class override
)

trainer.train()

This process can be very slow and prone to errors due to memory limitations.
If the process runs properly, you can see that several model checkpoints were stored in the indicated folder. You can inspect the contents and discover the best checkpoint in validation.
The, select that model version and run the inference in the dev and evl partitions.

In [ ]:
from transformers import pipeline, Wav2Vec2ForSequenceClassification, AutoFeatureExtractor
import os
import torch
import pickle
import pandas as pd # Ensure pandas is imported

# model_id is already defined from previous cells (f7ETePqLvQl) as 'w2v2base_age_regression_fe_freeze'
# DATADIR is already defined as '/content/drive/MyDrive/ets_data/'
# CWD is already defined 
# trainset is 'train_small'

checkpoint = 21 # Example checkpoint, adjust based on training results (was 100, then 1, changed to 21 to match existing file system checkpoint)

# The model was saved in the CWD, specifically in the output_dir defined in TrainingArguments.
# The actual path to the checkpoint folder will be CWD / model_id / checkpoint-XXX
model_path_to_load = os.path.join(CWD, model_id, f'checkpoint-{checkpoint}')

# The original base model name for the feature extractor
original_model_name = "facebook/wav2vec2-base"

# Explicitly load the fine-tuned model and its associated feature extractor
# Use Wav2Vec2ForSequenceClassification directly as it was used for training
model_loaded = Wav2Vec2ForSequenceClassification.from_pretrained(model_path_to_load, local_files_only=True)

# Load the feature extractor from the original base model, as it was not fine-tuned or saved with the checkpoint
feature_extractor_loaded = AutoFeatureExtractor.from_pretrained(original_model_name)


# Use 'audio-classification' pipeline, but it will output a score for regression
# Pass the loaded model and feature_extractor objects directly to the pipeline
classifier = pipeline(
    "audio-classification",
    model=model_loaded,
    feature_extractor=feature_extractor_loaded,
    device=0 if torch.cuda.is_available() else -1
)

# Re-define trainset as it was not in scope for this cell
trainset = 'train_small'

# Ensure the output directory for results exists on Google Drive
results_output_dir = f'{DATADIR}/{trainset}/models/{model_id}/'
if not os.path.isdir(results_output_dir):
    os.makedirs(results_output_dir, exist_ok=True)

for partition in ('dev', 'evl',):
    hyp, files, ref_labels = [], [], [] # Also collect reference labels for evaluation later

    # Use raw_audio_datasets for inference as it contains original audio paths
    # and labels, consistent with the fine-tuning setup
    current_raw_dataset = raw_audio_datasets[partition] # raw_audio_datasets from fBbvepQJLvQk

    for audio_sample in current_raw_dataset: # audio_sample will have 'audio', 'label', and 'fileid' keys
        # Access audio array and sampling rate using dictionary-like keys
        audio_array = audio_sample["audio"]["array"]
        sampling_rate = audio_sample["audio"]["sampling_rate"]
        file_id = audio_sample["fileid"] # Access the new 'fileid' feature

        # The pipeline returns a list of dictionaries with 'score' for regression
        # Pass the raw audio array and its sampling rate to the classifier
        prediction_result = classifier(audio_array, sampling_rate=sampling_rate)

        predicted_age = prediction_result[0]['score'] # Extract the score from the prediction
        hyp.append(predicted_age)

        files.append(file_id) # Use the directly stored file_id

        # Collect actual labels for evaluation
        ref_labels.append(audio_sample["label"])

    filename = f'{results_output_dir}/{partition}.pkl'

    # Save both hypotheses and reference labels
    pickle.dump({'hyp': hyp, 'fileids': files, 'ref': ref_labels}, open(filename, 'wb'))

print("Inference complete and results saved.")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Performance on the dev set
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
dev_res = pickle.load(open(filename, 'rb'))

hyp = dev_res['hyp']
ref = dev_res['ref'] # Use reference labels from the saved inference results

# Report the results for regression
print(f"Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}")
print(f"Mean Squared Error: {mean_squared_error(ref, hyp):.2f}")


And create the submission file:

In [ ]:
from pf_tools import create_submission_file

students_group = '00' # <--- CHANGE THIS ACCORDINGLY

model_id_short = model_id

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)

As mentioned, this process can be slow and difficult to "play with". For these reasons, this last part in completely optional.

# Contacts and support
You can contact the professors during the classes or the office hours.

Particularly, for this second laboratory assignment, you should contact Prof. Alberto Abad: alberto.abad@tecnico.ulisboa.pt


